In [1]:
%run ../config/bootstrap.py

### Libs

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from utils import get_project_root

In [3]:
project_root = get_project_root() 

L04AB04	adalimumab
L04AB05	certolizumab pegol
L04AB01	etanercept
L04AB06	golimumab
L04AB02	infliximab
L04AA24	abatacept
L01FA01	rituximab
L04AC07	tocilizumab

# Datasets

In [4]:
path = Path(project_root) / "data/02_silver/Medicamentos/dim_medicamentos_25092025.parquet"
df_medicamentos = pd.read_parquet(path) 
df_medicamentos = df_medicamentos[['IDENTIFICACAO_NOTIFICACAO','CODIGO_ATC','PRINCIPIOS_ATIVOS_WHODRUG']].drop_duplicates()
df_medicamentos.head()

,IDENTIFICACAO_NOTIFICACAO,CODIGO_ATC,PRINCIPIOS_ATIVOS_WHODRUG
0,BR-ANVISA-300000004,J01CF,OXACILLIN SODIUM
1,BR-ANVISA-300000005,N02BE,PARACETAMOL
2,BR-ANVISA-300000007,C03|N03AX|S01EC,ACETAZOLAMIDE SODIUM
3,BR-ANVISA-300000007,N03AB,PHENYTOIN
4,BR-ANVISA-300000008,C01DA|C05AE,GLYCERYL TRINITRATE


### Função

In [5]:
import re
import numpy as np
import pandas as pd
import unicodedata
from typing import Optional, List, Dict

def _strip_accents(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def make_nome_medicamento(
    df: pd.DataFrame,
    atc_col: str = "CODIGO_ATC",
    ativo_col: str = "PRINCIPIOS_ATIVOS_WHODRUG",
    saida_nome: str = "nome_medicamento",
    saida_flag: str = "medicamento_estudo",
    regras: Optional[List[Dict[str, str]]] = None,  # <-- compatível com Py<3.10
) -> pd.DataFrame:
    """
    Preenche `nome_medicamento` (string) e `medicamento_estudo` (bool) quando:
      (ATC começa com o código) E (princípio ativo contém o principio).
    Se várias regras casarem, a primeira na lista tem prioridade.
    """
    if not regras:
        raise ValueError("Forneça `regras` como lista de dicts: {'atc','principio','label'}.")

    # sanity checks mínimos
    faltando = {atc_col, ativo_col} - set(df.columns)
    if faltando:
        raise KeyError(f"Colunas ausentes no DataFrame: {faltando}")

    # normalização
    atc_norm = df[atc_col].astype(str).fillna("").str.lower().map(_strip_accents)
    ativo_norm = df[ativo_col].astype(str).fillna("").str.lower().map(_strip_accents)

    conditions, choices = [], []
    for r in regras:
        atc_code = _strip_accents(str(r["atc"]).lower())
        principio    = _strip_accents(str(r["principio"]).lower())
        label    = str(r["label"])

        # ATC: começa com (prefix match)
        mask_atc = atc_norm.str[:len(atc_code)].eq(atc_code)

        # ativo: contém
        principio_regex = re.compile(re.escape(principio))
        mask_ativo = ativo_norm.str.contains(principio_regex, na=False)

        conditions.append(mask_atc & mask_ativo)
        choices.append(label)

    df[saida_nome] = np.select(conditions, choices, default="outros")
    df[saida_flag] = df[saida_nome].ne("outros")
    return df


In [6]:
regras = [
    {"atc": "L04AB", "principio": "adalimumab",        "label": "L04AB04_adalimumab"},
    {"atc": "L04AB", "principio": "certolizumab pegol","label": "L04AB05_certolizumab_pegol"},
    {"atc": "L04AB", "principio": "etanercept",        "label": "L04AB01_etanercept"},
    {"atc": "L04AB", "principio": "golimumab",         "label": "L04AB06_golimumab"},
    {"atc": "L04AB", "principio": "infliximab",        "label": "L04AB02_infliximab"},
    {"atc": "L04AA", "principio": "abatacept",         "label": "L04AA24_abatacept"},
    {"atc": "L01FA", "principio": "rituximab",         "label": "L01FA01_rituximab"},
    {"atc": "L04AC", "principio": "tocilizumab",       "label": "L04AC07_tocilizumab"},
]


In [7]:
df_medicamentos = make_nome_medicamento(
    df_medicamentos,
    atc_col="CODIGO_ATC",
    ativo_col="PRINCIPIOS_ATIVOS_WHODRUG",
    saida_nome="nome_medicamento",
    saida_flag="medicamento_estudo",
    regras=regras,
)

# checar resultado
df_medicamentos["nome_medicamento"].value_counts(dropna=False).head(20)

nome_medicamento
outros                        480321
L04AB02_infliximab             10536
L04AB06_golimumab               3743
L01FA01_rituximab               3016
L04AB04_adalimumab              1624
L04AC07_tocilizumab              854
L04AB05_certolizumab_pegol       828
L04AB01_etanercept               374
L04AA24_abatacept                170
Name: count, dtype: int64

In [ ]:
df_medicamentos["medicamento_estudo"].value_counts(dropna=False).head(20)

medicamento_estudo
False    480321
True      21145
Name: count, dtype: int64

### Checks

In [23]:
df_medicamentos.head()

,IDENTIFICACAO_NOTIFICACAO,CODIGO_ATC,PRINCIPIOS_ATIVOS_WHODRUG,nome_medicamento,medicamento_estudo
0,BR-ANVISA-300000004,J01CF,OXACILLIN SODIUM,outros,False
1,BR-ANVISA-300000005,N02BE,PARACETAMOL,outros,False
2,BR-ANVISA-300000007,C03|N03AX|S01EC,ACETAZOLAMIDE SODIUM,outros,False
3,BR-ANVISA-300000007,N03AB,PHENYTOIN,outros,False
4,BR-ANVISA-300000008,C01DA|C05AE,GLYCERYL TRINITRATE,outros,False


In [10]:
df_medicamentos[df_medicamentos["CODIGO_ATC"].astype(str).str.contains("L04AB", na=False, case=False)
    &
    df_medicamentos["PRINCIPIOS_ATIVOS_WHODRUG"].astype(str).str.contains("adalimumab", na=False, case=False)].medicamento_estudo.value_counts(dropna=False)

medicamento_estudo
True    1624
Name: count, dtype: int64

In [11]:
df_medicamentos[
    df_medicamentos["CODIGO_ATC"].astype(str).str.contains("L04AB", na=False, case=False)
    &
    df_medicamentos["PRINCIPIOS_ATIVOS_WHODRUG"].astype(str).str.contains("certolizumab", na=False, case=False)
    ].medicamento_estudo.value_counts(dropna=False)

medicamento_estudo
True     828
False    393
Name: count, dtype: int64

In [12]:
df_medicamentos[
    df_medicamentos["CODIGO_ATC"].astype(str).str.contains("L04AB", na=False, case=False)
    &
    df_medicamentos["PRINCIPIOS_ATIVOS_WHODRUG"].astype(str).str.contains("etanercept", na=False, case=False)
    ].medicamento_estudo.value_counts(dropna=False)

medicamento_estudo
True    374
Name: count, dtype: int64

In [13]:
df_medicamentos[
    df_medicamentos["CODIGO_ATC"].astype(str).str.contains("L04AB", na=False, case=False)
    &
    df_medicamentos["PRINCIPIOS_ATIVOS_WHODRUG"].astype(str).str.contains("golimumab", na=False, case=False)
    ].medicamento_estudo.value_counts(dropna=False)

medicamento_estudo
True    3743
Name: count, dtype: int64

In [14]:
df_medicamentos[
    df_medicamentos["CODIGO_ATC"].astype(str).str.contains("L04AB", na=False, case=False)
    &
    df_medicamentos["PRINCIPIOS_ATIVOS_WHODRUG"].astype(str).str.contains("infliximab", na=False, case=False)
    ].medicamento_estudo.value_counts(dropna=False)

medicamento_estudo
True    10536
Name: count, dtype: int64

In [24]:
df_medicamentos[
    df_medicamentos["CODIGO_ATC"].astype(str).str.contains("L04AB", na=False, case=False)
    &
    df_medicamentos["PRINCIPIOS_ATIVOS_WHODRUG"].astype(str).str.contains("infliximab", na=False, case=False)
    ].PRINCIPIOS_ATIVOS_WHODRUG.value_counts(dropna=False)

PRINCIPIOS_ATIVOS_WHODRUG
INFLIXIMABE                5429
INFLIXIMAB                 5078
INFLIXIMAB BIOSIMILAR 1      14
INFLIXIMAB DYYB               9
INFLIXIMAB BIOSIMILAR 3       5
INFLIXIMAB BIOSIMILAR 2       1
Name: count, dtype: int64

In [16]:
df_medicamentos[
    df_medicamentos["CODIGO_ATC"].astype(str).str.contains("L04AA", na=False, case=False)
    &
    df_medicamentos["PRINCIPIOS_ATIVOS_WHODRUG"].astype(str).str.contains("abatacept", na=False, case=False)
    ].PRINCIPIOS_ATIVOS_WHODRUG.value_counts(dropna=False)

PRINCIPIOS_ATIVOS_WHODRUG
ABATACEPT     151
ABATACEPTE     19
Name: count, dtype: int64

In [17]:
df_medicamentos[
    df_medicamentos["CODIGO_ATC"].astype(str).str.contains("L04AA", na=False, case=False)
    &
    df_medicamentos["PRINCIPIOS_ATIVOS_WHODRUG"].astype(str).str.contains("abatacept", na=False, case=False)
    ].medicamento_estudo.value_counts(dropna=False)

medicamento_estudo
True    170
Name: count, dtype: int64

In [18]:
df_medicamentos[
    df_medicamentos["CODIGO_ATC"].astype(str).str.contains("L04AC", na=False, case=False)
    &
    df_medicamentos["PRINCIPIOS_ATIVOS_WHODRUG"].astype(str).str.contains("tocilizumab", na=False, case=False)
    ].medicamento_estudo.value_counts(dropna=False)

medicamento_estudo
True    854
Name: count, dtype: int64

In [19]:
df_medicamentos[
    df_medicamentos["CODIGO_ATC"].astype(str).str.contains("L01FA", na=False, case=False)
    &
    df_medicamentos["PRINCIPIOS_ATIVOS_WHODRUG"].astype(str).str.contains("rituximab", na=False, case=False)
    ].medicamento_estudo.value_counts(dropna=False)

medicamento_estudo
True    3016
Name: count, dtype: int64

## Notificacoes com MedEstudo

In [25]:
lista_notificacoes = df_medicamentos[df_medicamentos["medicamento_estudo"]==True].IDENTIFICACAO_NOTIFICACAO.unique().tolist()

# Reacoes

In [28]:
path = Path(project_root) / "data/02_silver/Reacoes/dim_reacoes_25092025.parquet"
df_reacoes = pd.read_parquet(path) 
df_reacoes = df_reacoes.drop_duplicates()
df_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,REACAO_EVTO_ADVERSO_MEDDRA_LLT,PT,HLT,HLGT,SOC,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,hash,current_flag,start_date,end_date
0,BR-ANVISA-300000004,COCEIRA,PRURIDO,PRURIDO NCO,QUADROS CLÍNICOS EPIDÉRMICOS E DÉRMICOS,DISTÚRBIOS DOS TECIDOS CUTÂNEOS E SUBCUTÂNEOS,NaT,NaT,3 DIA,NÃO,NONE,RECUPERADO/RESOLVIDO,5a12ce522274d3e218798b6179549068178c43ef28e4126ad996ead3d2ef5d96,True,2025-09-25 17:20:11.761986,NaT
1,BR-ANVISA-300000005,EDEMA PERIORBITAL,EDEMA PERIORBITAL,DISTÚRBIOS OCULARES NCO,TRANSTORNOS OCULARES NCO,DISTÚRBIOS OCULARES,2018-11-22,2018-11-22,NONE,SIM,OUTRO EFEITO CLINICAMENTE SIGNIFICATIVO,RECUPERADO/RESOLVIDO,ee8c54d3a9a13c5aa00244f5fe7a38bb00f95cdc4c605b65b9e653ae7e399e97,True,2025-09-25 17:20:11.761986,NaT
2,BR-ANVISA-300000007,EXANTEMA ALÉRGICO,DERMATITE ALÉRGICA,DERMATITE E ECZEMA,QUADROS CLÍNICOS EPIDÉRMICOS E DÉRMICOS,DISTÚRBIOS DOS TECIDOS CUTÂNEOS E SUBCUTÂNEOS,2018-11-15,NaT,2 DIA,SIM,OUTRO EFEITO CLINICAMENTE SIGNIFICATIVO,RECUPERADO/RESOLVIDO,d0cec3307f1184ca773ad2a2710aa9e8442e7e79d0f06c421fa7bf5e8d0b08a5,True,2025-09-25 17:20:11.761986,NaT
3,BR-ANVISA-300000008,FLEBITE,FLEBITE,FLEBITE NCO,INFECÇÕES E INFLAMAÇÕES VASCULARES,DISTÚRBIOS VASCULARES,2018-10-25,NaT,5 DIA,SIM,OUTRO EFEITO CLINICAMENTE SIGNIFICATIVO,RECUPERADO/RESOLVIDO,e67bc5e469f0e80755b4c00c16782ca0530549a70b9f89c7cf28119614178337,True,2025-09-25 17:20:11.761986,NaT
4,BR-ANVISA-300000010,PARESTESIA,PARESTESIA,PARESTESIAS E DISESTESIAS,DISTÚRBIOS NEUROLÓGICOS NCO,DISTÚRBIOS DO SISTEMA NERVOSO,NaT,NaT,NONE,SIM,HOSPITALIZAÇÃO/PROLONGAMENTO DE HOSPITALIZAÇÃO,RECUPERADO/RESOLVIDO,b1afde1e9d079d78ccf9d6c0dbce3085c9719083b5bfa8d5a735608a895c2c71,True,2025-09-25 17:20:11.761986,NaT


# Desproporcionalidade

In [29]:
import math
from typing import Iterable, Optional
import pandas as pd
import numpy as np

def tabela_desproporcionalidade(
    df_medicamentos: pd.DataFrame,
    df_reacoes: pd.DataFrame,
    *,
    id_col: str = "IDENTIFICACAO_NOTIFICACAO",
    med_name_col: str = "nome_medicamento",
    med_flag_col: str = "medicamento_estudo",
    reaction_col: str = "PT",           # opções comuns: "PT", "HLT", "HLGT", "SOC", "REACAO_EVTO_ADVERSO_MEDDRA_LLT"
    soc_col: Optional[str] = "SOC",     # usado apenas para enriquecer a saída; pode ser None
    only_medicamento_estudo: bool = True,
    target_medicamentos: Optional[Iterable[str]] = None,   # lista explícita de medicamentos; se None pega todos (ou apenas os de estudo)
    reacoes_current_flag_col: Optional[str] = None,        # ex.: "current_flag" para filtrar reações vigentes
    min_cases: int = 3,
    apply_continuity_correction: bool = True,
    dropna_reaction: bool = True,
) -> pd.DataFrame:
    """
    Cria a Tabela 1 de desproporcionalidade (ROR e IC95%) por medicamento x reação.

    Lógica (2x2 por medicamento m e evento e):
        a = nº de notificações COM o medicamento m E COM o evento e
        b = nº de notificações COM o medicamento m E SEM o evento e
        c = nº de notificações SEM o medicamento m E COM o evento e
        d = nº de notificações SEM o medicamento m E SEM o evento e

        ROR = (a*d) / (b*c)
        se = sqrt(1/a + 1/b + 1/c + 1/d)
        IC95% = exp[ ln(ROR) ± 1.96 * se ]

    Observações importantes:
    - Deduplicamos por (id_col, reaction_col) para não contar o mesmo evento mais de uma vez na mesma notificação.
    - Universo de comparação ("todos os outros fármacos") = todas as notificações presentes em df_reacoes que NÃO contêm o medicamento m.
    - Se qualquer célula de (a,b,c,d) for zero, aplica-se correção de continuidade (Haldane–Anscombe) somando 0.5 a todas as células
      para evitar divisão por zero e ICs indefinidos (configurável via apply_continuity_correction).

    Retorna:
        DataFrame em formato longo contendo, para cada medicamento e evento:
            [medicamento, SOC, evento, a,b,c,d, N_medicamento, N_outros, ROR, IC95_low, IC95_high, sinal]
    """
    df_meds = df_medicamentos.copy()
    df_rx = df_reacoes.copy()

    # Filtrar medicamentos de interesse (se solicitado)
    if only_medicamento_estudo and med_flag_col in df_meds.columns:
        df_meds = df_meds[df_meds[med_flag_col] == True].copy()

    if target_medicamentos is not None:
        df_meds = df_meds[df_meds[med_name_col].isin(list(target_medicamentos))].copy()

    # Garantir colunas mínimas
    required_meds_cols = {id_col, med_name_col}
    required_rx_cols = {id_col, reaction_col}
    missing_meds = required_meds_cols - set(df_meds.columns)
    missing_rx = required_rx_cols - set(df_rx.columns)
    if missing_meds:
        raise ValueError(f"df_medicamentos está sem as colunas obrigatórias: {missing_meds}")
    if missing_rx:
        raise ValueError(f"df_reacoes está sem as colunas obrigatórias: {missing_rx}")

    # Filtrar reações "atuais" se fornecido o flag SCD2
    if reacoes_current_flag_col and (reacoes_current_flag_col in df_rx.columns):
        df_rx = df_rx[df_rx[reacoes_current_flag_col] == True].copy()

    # Normalizar o universo de reações: (id, evento) único
    if dropna_reaction:
        df_rx = df_rx[~df_rx[reaction_col].isna()].copy()

    df_rx_pairs = df_rx[[id_col, reaction_col]].drop_duplicates()

    # (Opcional) mapa evento -> SOC para enriquecer a saída
    soc_map = {}
    if soc_col and (soc_col in df_rx.columns):
        soc_map = (
            df_rx[[reaction_col, soc_col]]
            .dropna(subset=[reaction_col])
            .drop_duplicates(subset=[reaction_col])
            .set_index(reaction_col)[soc_col]
            .to_dict()
        )

    # Universo de notificações consideradas
    all_ids = set(df_rx_pairs[id_col].unique())

    # Conjunto de eventos existentes
    all_events = df_rx_pairs[reaction_col].unique().tolist()

    # Índice invertido: evento -> set(ids que têm o evento)
    event_to_ids = (
        df_rx_pairs.groupby(reaction_col)[id_col]
        .agg(lambda s: set(s.unique()))
        .to_dict()
    )

    # Índice invertido: medicamento -> set(ids que contêm o medicamento)
    med_to_ids = (
        df_meds.groupby(med_name_col)[id_col]
        .agg(lambda s: set(s.unique()))
        .to_dict()
    )

    # Se nada sobrou (ex.: filtro muito restrito), aborta cedo
    if not med_to_ids:
        return pd.DataFrame(
            columns=[
                "medicamento", "SOC", "evento", "a", "b", "c", "d",
                "N_medicamento", "N_outros", "ROR", "IC95_low", "IC95_high", "sinal"
            ]
        )

    rows = []
    for med, exposed_ids in med_to_ids.items():
        # Notificações com e sem o medicamento
        exposed_ids = set(exposed_ids) & all_ids
        non_exposed_ids = all_ids - exposed_ids
        N1 = len(exposed_ids)
        N0 = len(non_exposed_ids)
        if N1 == 0 or N0 == 0:
            # Sem universo de comparação
            continue

        # Para eficiência, pré-computar máscara dos ids expostos
        for ev in all_events:
            ids_with_ev = event_to_ids.get(ev, set())
            # a = expostos com evento
            a = len(exposed_ids & ids_with_ev)
            # c = não expostos com evento
            c = len(non_exposed_ids & ids_with_ev)
            # b = expostos sem evento
            b = N1 - a
            # d = não expostos sem evento
            d = N0 - c

            # Pular eventos que não aparecem em ninguém
            if (a + b + c + d) == 0:
                continue

            # Correção de continuidade se houver zero em alguma célula
            A, B, C, D = a, b, c, d
            if apply_continuity_correction and (0 in (a, b, c, d)):
                A = a + 0.5
                B = b + 0.5
                C = c + 0.5
                D = d + 0.5

            # Calcular ROR e IC95%
            try:
                ror = (A * D) / (B * C)
                se = math.sqrt(1.0 / A + 1.0 / B + 1.0 / C + 1.0 / D)
                log_ror = math.log(ror)
                ci_low = math.exp(log_ror - 1.96 * se)
                ci_high = math.exp(log_ror + 1.96 * se)
            except Exception:
                ror = np.nan
                ci_low = np.nan
                ci_high = np.nan

            # Critério de sinal (mínimo de casos + IC95% inferior > 1)
            sinal = (a >= min_cases) and (pd.notna(ci_low) and (ci_low > 1.0))

            rows.append({
                "medicamento": med,
                "SOC": soc_map.get(ev) if soc_map else None,
                "evento": ev,
                "a": a, "b": b, "c": c, "d": d,
                "N_medicamento": N1,
                "N_outros": N0,
                "ROR": ror,
                "IC95_low": ci_low,
                "IC95_high": ci_high,
                "sinal": sinal,
            })

    out = pd.DataFrame(rows)
    # Ordenação útil: por medicamento, sinal prioritário, depois ROR desc e a desc
    if not out.empty:
        out = out.sort_values(
            by=["medicamento", "sinal", "ROR", "a"],
            ascending=[True, False, False, False],
            kind="stable"
        ).reset_index(drop=True)

    return out


In [ ]:
# 1) Você já tem df_medicamentos após make_nome_medicamento(...)
# 2) Carregou df_reacoes, removeu duplicados etc.

t1 = tabela_desproporcionalidade(
    df_medicamentos=df_medicamentos,
    df_reacoes=df_reacoes,
    id_col="",
    med_name_col="nome_medicamento",
    med_flag_col="medicamento_estudo",
    reaction_col="PT",     # troque para "SOC", "HLT", "HLGT" ou "REACAO_EVTO_ADVERSO_MEDDRA_LLT" se preferir
    soc_col="SOC",         # mantém a coluna SOC na saída (se existir em df_reacoes)
    only_medicamento_estudo=True,  # calcula apenas para os medicamentos marcados no estudo
    target_medicamentos=None,      # ou passe uma lista específica, ex.: ["L04AB02_infliximab", "L04AB06_golimumab"]
    reacoes_current_flag_col="current_flag",  # usa apenas reações vigentes (True); remova se não quiser filtrar
    min_cases=3,
    apply_continuity_correction=True,
)

# Visualizar top sinais por medicamento
t1.query("sinal == True").head(20)


,medicamento,SOC,evento,a,b,c,d,N_medicamento,N_outros,ROR,IC95_low,IC95_high,sinal
0,L01FA01_rituximab,"NEOPLASIAS BENIGNAS, MALIGNAS E NÃO ESPECIFICADAS (INCL. CISTOS E PÓLIPOS)",SÍNDROME DE RICHTER,6,2986,3,311525,2992,311528,208.657066,52.158871,834.714605,True
1,L01FA01_rituximab,DISTÚRBIOS MUSCULOESQUELÉTICOS E DO TECIDO CONJUNTIVO,POLIMIOSITE,3,2989,6,311522,2992,311528,52.111408,13.026511,208.467100,True
2,L01FA01_rituximab,DISTÚRBIOS MUSCULOESQUELÉTICOS E DO TECIDO CONJUNTIVO,ESCLERODERMIA SISTÊMICA,8,2984,18,311510,2992,311528,46.397081,20.158313,106.789150,True
3,L01FA01_rituximab,INFECÇÕES E INFESTAÇÕES,FASCIÍTE NECROSANTE,3,2989,7,311521,2992,311528,44.666778,11.544648,172.817842,True
4,L01FA01_rituximab,"NEOPLASIAS BENIGNAS, MALIGNAS E NÃO ESPECIFICADAS (INCL. CISTOS E PÓLIPOS)",DISTÚRBIO LINFOPROLIFERATIVO,3,2989,7,311521,2992,311528,44.666778,11.544648,172.817842,True
5,L01FA01_rituximab,INVESTIGAÇÕES,PRESSÃO ARTERIAL SISTÓLICA AUMENTADA,8,2984,19,311509,2992,311528,43.954988,19.226915,100.486272,True
6,L01FA01_rituximab,DISTÚRBIOS DO OUVIDO E DO LABIRINTO,PRURIDO DO OUVIDO,18,2974,43,311485,2992,311528,43.842996,25.259273,76.099113,True
7,L01FA01_rituximab,DISTÚRBIOS MUSCULOESQUELÉTICOS E DO TECIDO CONJUNTIVO,ESCLERODERMIA,3,2989,9,311519,2992,311528,34.740604,9.400342,128.389968,True
8,L01FA01_rituximab,DISTÚRBIOS DOS SISTEMAS HEMATOLÓGICO E LINFÁTICO,ANEMIA HEMOLÍTICA AUTOIMUNE,3,2989,11,311517,2992,311528,28.423948,7.925612,101.937969,True
9,L01FA01_rituximab,DISTÚRBIOS DOS TECIDOS CUTÂNEOS E SUBCUTÂNEOS,DERMATOMIOSITE,4,2988,16,311512,2992,311528,26.063588,8.708275,78.007480,True


In [31]:

t1.medicamento.unique()

array(['L01FA01_rituximab', 'L04AA24_abatacept', 'L04AB01_etanercept',
       'L04AB02_infliximab', 'L04AB04_adalimumab',
       'L04AB05_certolizumab_pegol', 'L04AB06_golimumab',
       'L04AC07_tocilizumab'], dtype=object)

# Teste 2

In [34]:
import math
from typing import Iterable, Optional
import pandas as pd
import numpy as np

def tabela_desproporcionalidade(
    df_medicamentos: pd.DataFrame,
    df_reacoes: pd.DataFrame,
    *,
    id_col: str = "IDENTIFICACAO_NOTIFICACAO",
    med_name_col: str = "nome_medicamento",
    med_flag_col: str = "medicamento_estudo",
    reaction_col: str = "PT",           # "PT", "HLT", "HLGT", "SOC", "REACAO_EVTO_ADVERSO_MEDDRA_LLT"
    soc_col: Optional[str] = "SOC",
    only_medicamento_estudo: bool = True,
    target_medicamentos: Optional[Iterable[str]] = None,
    reacoes_current_flag_col: Optional[str] = None,  # ex.: "current_flag"
    min_cases: int = 3,
    apply_continuity_correction: bool = True,
    dropna_reaction: bool = True,
) -> pd.DataFrame:
    """
    Tabela 1 de desproporcionalidade (ROR e IC95%) por medicamento x reação, com DISTINCTCOUNT
    de IDENTIFICACAO_NOTIFICACAO na base de reações.
    """

    df_meds = df_medicamentos.copy()
    df_rx = df_reacoes.copy()

    # 1) Filtro de medicamentos de interesse
    if only_medicamento_estudo and med_flag_col in df_meds.columns:
        df_meds = df_meds[df_meds[med_flag_col] == True].copy()
    if target_medicamentos is not None:
        df_meds = df_meds[df_meds[med_name_col].isin(list(target_medicamentos))].copy()

    # 2) Checagem de colunas
    required_meds_cols = {id_col, med_name_col}
    required_rx_cols = {id_col, reaction_col}
    missing_meds = required_meds_cols - set(df_meds.columns)
    missing_rx = required_rx_cols - set(df_rx.columns)
    if missing_meds:
        raise ValueError(f"df_medicamentos está sem as colunas obrigatórias: {missing_meds}")
    if missing_rx:
        raise ValueError(f"df_reacoes está sem as colunas obrigatórias: {missing_rx}")

    # 3) Filtrar reações "atuais" se houver coluna SCD2
    if reacoes_current_flag_col and (reacoes_current_flag_col in df_rx.columns):
        df_rx = df_rx[df_rx[reacoes_current_flag_col] == True].copy()

    # 4) (Opcional) remover NaN do evento
    if dropna_reaction:
        df_rx = df_rx[~df_rx[reaction_col].isna()].copy()

    # 5) DISTINCTCOUNT no universo de reações:
    #    - Para evento: usar pares (id, evento) distintos
    #    - Para universo total: usar id distintos (distinctcount de id_col)
    df_rx_pairs = df_rx[[id_col, reaction_col]].drop_duplicates()
    df_rx_ids = df_rx[[id_col]].drop_duplicates()  # <-- DISTINCTCOUNT do id_col
    all_ids = set(df_rx_ids[id_col].tolist())

    # 6) (Opcional) mapa evento -> SOC para enriquecer saída
    soc_map = {}
    if soc_col and (soc_col in df_rx.columns):
        soc_map = (
            df_rx[[reaction_col, soc_col]]
            .dropna(subset=[reaction_col])
            .drop_duplicates(subset=[reaction_col])
            .set_index(reaction_col)[soc_col]
            .to_dict()
        )

    # 7) Conjunto de eventos existentes e índices invertidos
    all_events = df_rx_pairs[reaction_col].unique().tolist()
    event_to_ids = (
        df_rx_pairs.groupby(reaction_col)[id_col]
        .agg(lambda s: set(s.unique()))
        .to_dict()
    )

    # Medicamento -> ids de notificações (distinct por id)
    med_to_ids = (
        df_meds.groupby(med_name_col)[id_col]
        .agg(lambda s: set(s.unique()))
        .to_dict()
    )
    if not med_to_ids:
        return pd.DataFrame(
            columns=[
                "medicamento", "SOC", "evento", "a", "b", "c", "d",
                "N_medicamento", "N_outros", "ROR", "IC95_low", "IC95_high", "sinal"
            ]
        )

    rows = []
    for med, exposed_ids in med_to_ids.items():
        exposed_ids = set(exposed_ids) & all_ids          # apenas ids que existem na base de reações
        non_exposed_ids = all_ids - exposed_ids
        N1 = len(exposed_ids)
        N0 = len(non_exposed_ids)
        if N1 == 0 or N0 == 0:
            continue

        for ev in all_events:
            ids_with_ev = event_to_ids.get(ev, set())

            # 2x2 com DISTINCTCOUNT de ids
            a = len(exposed_ids & ids_with_ev)   # expostos com evento
            c = len(non_exposed_ids & ids_with_ev) # não expostos com evento
            b = N1 - a                            # expostos sem evento
            d = N0 - c                            # não expostos sem evento

            if (a + b + c + d) == 0:
                continue

            # Correção de continuidade quando houver zeros
            A, B, C, D = a, b, c, d
            if apply_continuity_correction and (0 in (a, b, c, d)):
                A = a + 0.5
                B = b + 0.5
                C = c + 0.5
                D = d + 0.5

            try:
                ror = (A * D) / (B * C)
                se = math.sqrt(1.0 / A + 1.0 / B + 1.0 / C + 1.0 / D)
                log_ror = math.log(ror)
                ci_low = math.exp(log_ror - 1.96 * se)
                ci_high = math.exp(log_ror + 1.96 * se)
            except Exception:
                ror = np.nan
                ci_low = np.nan
                ci_high = np.nan

            sinal = (a >= min_cases) and (pd.notna(ci_low) and (ci_low > 1.0))

            rows.append({
                "medicamento": med,
                "SOC": soc_map.get(ev) if soc_map else None,
                "evento": ev,
                "a": a, "b": b, "c": c, "d": d,
                "N_medicamento": N1,
                "N_outros": N0,
                "ROR": ror,
                "IC95_low": ci_low,
                "IC95_high": ci_high,
                "sinal": sinal,
            })

    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(
            by=["medicamento", "sinal", "ROR", "a"],
            ascending=[True, False, False, False],
            kind="stable"
        ).reset_index(drop=True)

    return out


In [ ]:
t1 = tabela_desproporcionalidade(
    df_medicamentos=df_medicamentos,
    df_reacoes=df_reacoes,
    id_col="IDENTIFICACAO_NOTIFICACAO",
    med_name_col="nome_medicamento",
    med_flag_col="medicamento_estudo",
    reaction_col="PT",     # troque para "SOC", "HLT", "HLGT" ou "REACAO_EVTO_ADVERSO_MEDDRA_LLT" se preferir
    soc_col="SOC",         # mantém a coluna SOC na saída (se existir em df_reacoes)
    only_medicamento_estudo=True,  # calcula apenas para os medicamentos marcados no estudo
    target_medicamentos=None,      # ou passe uma lista específica, ex.: ["L04AB02_infliximab", "L04AB06_golimumab"]
    reacoes_current_flag_col="current_flag",  # usa apenas reações vigentes (True); remova se não quiser filtrar
    min_cases=3,
    apply_continuity_correction=True,
)

# Visualizar top sinais por medicamento
t1.query("sinal == True").head(20)


,medicamento,SOC,evento,a,b,c,d,N_medicamento,N_outros,ROR,IC95_low,IC95_high,sinal
0,L01FA01_rituximab,"NEOPLASIAS BENIGNAS, MALIGNAS E NÃO ESPECIFICADAS (INCL. CISTOS E PÓLIPOS)",SÍNDROME DE RICHTER,6,2986,3,311525,2992,311528,208.657066,52.158871,834.714605,True
1,L01FA01_rituximab,DISTÚRBIOS MUSCULOESQUELÉTICOS E DO TECIDO CONJUNTIVO,POLIMIOSITE,3,2989,6,311522,2992,311528,52.111408,13.026511,208.467100,True
2,L01FA01_rituximab,DISTÚRBIOS MUSCULOESQUELÉTICOS E DO TECIDO CONJUNTIVO,ESCLERODERMIA SISTÊMICA,8,2984,18,311510,2992,311528,46.397081,20.158313,106.789150,True
3,L01FA01_rituximab,INFECÇÕES E INFESTAÇÕES,FASCIÍTE NECROSANTE,3,2989,7,311521,2992,311528,44.666778,11.544648,172.817842,True
4,L01FA01_rituximab,"NEOPLASIAS BENIGNAS, MALIGNAS E NÃO ESPECIFICADAS (INCL. CISTOS E PÓLIPOS)",DISTÚRBIO LINFOPROLIFERATIVO,3,2989,7,311521,2992,311528,44.666778,11.544648,172.817842,True
5,L01FA01_rituximab,INVESTIGAÇÕES,PRESSÃO ARTERIAL SISTÓLICA AUMENTADA,8,2984,19,311509,2992,311528,43.954988,19.226915,100.486272,True
6,L01FA01_rituximab,DISTÚRBIOS DO OUVIDO E DO LABIRINTO,PRURIDO DO OUVIDO,18,2974,43,311485,2992,311528,43.842996,25.259273,76.099113,True
7,L01FA01_rituximab,DISTÚRBIOS MUSCULOESQUELÉTICOS E DO TECIDO CONJUNTIVO,ESCLERODERMIA,3,2989,9,311519,2992,311528,34.740604,9.400342,128.389968,True
8,L01FA01_rituximab,DISTÚRBIOS DOS SISTEMAS HEMATOLÓGICO E LINFÁTICO,ANEMIA HEMOLÍTICA AUTOIMUNE,3,2989,11,311517,2992,311528,28.423948,7.925612,101.937969,True
9,L01FA01_rituximab,DISTÚRBIOS DOS TECIDOS CUTÂNEOS E SUBCUTÂNEOS,DERMATOMIOSITE,4,2988,16,311512,2992,311528,26.063588,8.708275,78.007480,True


In [37]:
t1.query("medicamento == 'L01FA01_rituximab' and evento == 'IRRITAÇÃO DA GARGANTA'")


,medicamento,SOC,evento,a,b,c,d,N_medicamento,N_outros,ROR,IC95_low,IC95_high,sinal
14,L01FA01_rituximab,"DISTÚRBIOS RESPIRATÓRIOS, TORÁCICOS E DO MEDIASTINO",IRRITAÇÃO DA GARGANTA,270,2722,1524,310004,2992,311528,20.177064,17.632447,23.088907,True


In [38]:
t1.query("medicamento == 'L01FA01_rituximab'").sort_values(by="a", ascending=False).head(20)

,medicamento,SOC,evento,a,b,c,d,N_medicamento,N_outros,ROR,IC95_low,IC95_high,sinal
87,L01FA01_rituximab,DISTÚRBIOS DOS TECIDOS CUTÂNEOS E SUBCUTÂNEOS,PRURIDO,576,2416,30437,281091,2992,311528,2.201763,2.008946,2.413087,True
77,L01FA01_rituximab,"DISTÚRBIOS RESPIRATÓRIOS, TORÁCICOS E DO MEDIASTINO",DISPNEIA,314,2678,12523,299005,2992,311528,2.799556,2.487275,3.151044,True
38,L01FA01_rituximab,DISTÚRBIOS DO SISTEMA NERVOSO,TREMOR,284,2708,5459,306069,2992,311528,5.879981,5.188305,6.663868,True
14,L01FA01_rituximab,"DISTÚRBIOS RESPIRATÓRIOS, TORÁCICOS E DO MEDIASTINO",IRRITAÇÃO DA GARGANTA,270,2722,1524,310004,2992,311528,20.177064,17.632447,23.088907,True
69,L01FA01_rituximab,DISTÚRBIOS VASCULARES,RUBOR,212,2780,7650,303878,2992,311528,3.029207,2.629563,3.489588,True
107,L01FA01_rituximab,DISTÚRBIOS GASTROINTESTINAIS,NÁUSEA,210,2782,15680,295848,2992,311528,1.424245,1.236717,1.640209,True
62,L01FA01_rituximab,"DISTÚRBIOS RESPIRATÓRIOS, TORÁCICOS E DO MEDIASTINO",TOSSE,207,2785,6680,304848,2992,311528,3.391970,2.939218,3.914464,True
27,L01FA01_rituximab,"LESÕES, INTOXICAÇÕES E COMPLICAÇÕES DE PROCEDIMENTOS",REAÇÃO RELACIONADA A INFUSÃO,163,2829,1698,309830,2992,311528,10.513333,8.914841,12.398446,True
8256,L01FA01_rituximab,DISTÚRBIOS DO SISTEMA NERVOSO,CEFALEIA,149,2843,19984,291544,2992,311528,0.764594,0.648071,0.902069,False
75,L01FA01_rituximab,DISTÚRBIOS CARDÍACOS,TAQUICARDIA,148,2844,5678,305850,2992,311528,2.803143,2.371255,3.313693,True


In [39]:
t1.query("medicamento == 'L04AB02_infliximab'").sort_values(by="a", ascending=False).head(20)

,medicamento,SOC,evento,a,b,c,d,N_medicamento,N_outros,ROR,IC95_low,IC95_high,sinal
27086,L04AB02_infliximab,DISTÚRBIOS GASTROINTESTINAIS,DOENÇA DE CROHN,1910,8618,782,303210,10528,303992,85.933726,78.858587,93.643643,True
27220,L04AB02_infliximab,"LESÕES, INTOXICAÇÕES E COMPLICAÇÕES DE PROCEDIMENTOS",USO NÃO DESCRITO EM BULA (OFF LABEL),1813,8715,7323,296669,10528,303992,8.427787,7.971567,8.910116,True
27304,L04AB02_infliximab,DISTÚRBIOS GERAIS E QUADROS CLÍNICOS NO LOCAL DE ADMINISTRAÇÃO,MEDICAMENTO INEFICAZ,1288,9240,9090,294902,10528,303992,4.522283,4.250759,4.811151,True
27078,L04AB02_infliximab,"LESÕES, INTOXICAÇÕES E COMPLICAÇÕES DE PROCEDIMENTOS",OMISSÃO INTENCIONAL DA DOSE,870,9658,204,303788,10528,303992,134.144385,115.020118,156.448423,True
35112,L04AB02_infliximab,DISTÚRBIOS DOS TECIDOS CUTÂNEOS E SUBCUTÂNEOS,PRURIDO,647,9881,30366,273626,10528,303992,0.590029,0.544438,0.639437,False
27099,L04AB02_infliximab,DISTÚRBIOS GASTROINTESTINAIS,COLITE ULCERATIVA,566,9962,371,303621,10528,303992,46.497306,40.729583,53.081798,True
27497,L04AB02_infliximab,"DISTÚRBIOS RESPIRATÓRIOS, TORÁCICOS E DO MEDIASTINO",DISPNEIA,562,9966,12275,291717,10528,303992,1.340157,1.228645,1.461789,True
27423,L04AB02_infliximab,"LESÕES, INTOXICAÇÕES E COMPLICAÇÕES DE PROCEDIMENTOS",PROBLEMA RELACIONADO À OMISSÃO DE DOSE DO PRODUTO,526,10002,6013,297979,10528,303992,2.606114,2.378667,2.855308,True
27498,L04AB02_infliximab,DISTÚRBIOS GERAIS E QUADROS CLÍNICOS NO LOCAL DE ADMINISTRAÇÃO,DOR,453,10075,10235,293757,10528,303992,1.290487,1.172153,1.420767,True
27208,L04AB02_infliximab,"LESÕES, INTOXICAÇÕES E COMPLICAÇÕES DE PROCEDIMENTOS",REAÇÃO RELACIONADA A INFUSÃO,428,10100,1433,302559,10528,303992,8.947182,8.017031,9.985251,True
